# RAG Evaluation and Building a Production RAG Chatbot

**Prerequisites:** We have completed Notebook 1 (Basic RAG) and Notebook 2 (Advanced RAG patterns, chunking, hybrid search, reranking).

**What this notebook covers:**

1. Why evaluation matters in RAG
2. RAGAS - Automated RAG evaluation framework
   - Faithfulness
   - Answer Relevancy
   - Context Precision
   - Context Recall
3. DeepEval - Production LLM testing framework
   - Hallucination metric
   - Answer Relevancy metric
   - Faithfulness metric
   - Writing test cases
4. Building a Full RAG Chatbot
   - Persistent vector store (save/load)
   - Multi-document ingestion
   - Streaming responses
   - Interactive terminal chat loop

---

**Models:**
- LLM: `llama-3.1-8b-instant` via Groq
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2` 
- RAGAS judge LLM: Groq 

## 1. Setup

In [50]:
# # Install all dependencies for this notebook.
# # ragas: RAG evaluation framework by Exploding Gradients
# # deepeval: production LLM testing by Confident AI

# !pip install langchain langchain-groq langchain-community langchain-huggingface sentence-transformers faiss-cpu rank_bm25 ragas deepeval python-dotenv
# !pip install langchain_google_genai

In [ ]:
# Loading environment and initialize models

from dotenv import load_dotenv
import os

load_dotenv()

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=1000,
)

# Local embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Models initialized.")

d:\Work\Learning\Topics_by_Abhay_Sir\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1351.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models initialized.


In [2]:
# Build a knowledge base that we will use throughout this notebook for evaluation.
# We use a more structured corpus here so we can ask precise evaluation questions.

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

knowledge_base_text = """
Retrieval-Augmented Generation (RAG) is an AI framework that retrieves relevant information
from an external knowledge base before generating a response. RAG was introduced by Lewis et al.
in 2020 in the paper 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks'.
The core idea is to augment a frozen language model with a retrieval component that fetches
relevant documents at inference time.

RAG reduces hallucination by grounding the model's response in factual retrieved context.
Instead of relying solely on parametric knowledge (weights), the model uses retrieved documents
as grounding evidence. This allows the model to cite sources and produce verifiable outputs.
Hallucination is the tendency of LLMs to generate plausible-sounding but factually incorrect information.

The RAG pipeline consists of four main components: the document store, the retriever, the reader,
and the generator. The document store indexes documents for retrieval. The retriever finds relevant
documents using embedding similarity or keyword search. The reader processes retrieved documents.
The generator uses the retrieved context to produce the final answer.

FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta Research.
It enables efficient similarity search over large collections of dense vectors. FAISS supports
both exact and approximate nearest neighbor search. It is widely used in RAG systems as the
vector store backend due to its speed and scalability. FAISS can handle billions of vectors.

Chunking is the process of splitting documents into smaller pieces before embedding them.
Common chunking strategies include fixed-size chunking, recursive character splitting, semantic
chunking, and hierarchical chunking. Chunk size and overlap are hyperparameters that significantly
affect retrieval quality. Too small chunks lose context; too large chunks reduce precision.

Embeddings are dense vector representations of text that capture semantic meaning. Similar texts
have embeddings that are close in vector space, allowing cosine similarity or dot product to measure
relevance. Popular embedding models include Sentence-BERT, OpenAI text-embedding-ada-002, and
BGE (BAAI General Embedding). The all-MiniLM-L6-v2 model produces 384-dimensional embeddings.

Reranking is a post-retrieval technique that uses a cross-encoder model to rescore retrieved
documents. Unlike bi-encoders that embed query and document separately, cross-encoders process
the (query, document) pair together, enabling richer interaction and more accurate relevance scoring.
Reranking improves precision by filtering out low-quality initial retrievals.

Hybrid search combines dense retrieval (semantic vector search) with sparse retrieval (keyword
matching like BM25). Dense methods capture semantic meaning while sparse methods handle exact
keyword matches. Reciprocal Rank Fusion (RRF) is commonly used to merge the ranked lists from
both methods into a single unified ranking.

RAGAS (Retrieval-Augmented Generation Assessment) is an evaluation framework for RAG systems.
It provides reference-free metrics: faithfulness measures if the answer is grounded in context,
answer relevancy measures if the answer addresses the question, context precision measures if
retrieved context is relevant, and context recall measures if all relevant information was retrieved.

LangChain is a framework for building applications powered by language models. It provides
abstractions for chains, agents, retrievers, and memory. LangChain integrates with many LLM
providers (OpenAI, Anthropic, Groq) and vector stores (FAISS, Chroma, Pinecone). It simplifies
building RAG pipelines by providing ready-made components that can be composed together.
"""

# Split and embed
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
raw_doc = Document(page_content=knowledge_base_text, metadata={"source": "rag_knowledge_base"})
chunks = splitter.split_documents([raw_doc])

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Knowledge base ready. {len(chunks)} chunks indexed in FAISS.")

Knowledge base ready. 11 chunks indexed in FAISS.


In [4]:
# Define the base RAG function we will evaluate throughout this notebook.
# This function returns both the answer AND the retrieved context - both are needed for evaluation.

def rag_pipeline(query, k=3):
    """
    Runs the RAG pipeline and returns the answer, retrieved context, and source docs.
    Evaluation frameworks need access to:
    - The original query
    - The retrieved context (for faithfulness and context metrics)
    - The generated answer (for relevancy metrics)
    """
    # Retrieve
    docs = vectorstore.similarity_search(query, k=k)
    context_texts = [doc.page_content for doc in docs]
    context = "\n\n".join(context_texts)
    
    # Generate
    messages = [
        SystemMessage(content=(
            "You are a helpful assistant. Answer questions using only the provided context. "
            "If the answer is not in the context, say 'I don't know based on the provided information.'"
        )),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ]
    response = llm.invoke(messages)
    
    return {
        "question": query,
        "answer": response.content,
        "contexts": context_texts,   # List of retrieved chunk texts
        "source_docs": docs
    }


# Quick test
result = rag_pipeline("What is FAISS and who created it?")
print(f"Question: {result['question']}")
print(f"Source docs: {result['source_docs']}")
print(f"Answer: {result['answer']}")
print(f"Number of context chunks: {len(result['contexts'])}")

Question: What is FAISS and who created it?
Source docs: [Document(id='c791efb1-bf15-4131-8843-413b31a2685e', metadata={'source': 'rag_knowledge_base', 'chunk_id': 4}, page_content='FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta Research.\nIt enables efficient similarity search over large collections of dense vectors. FAISS supports\nboth exact and approximate nearest neighbor search. It is widely used in RAG systems as the\nvector store backend due to its speed and scalability. FAISS can handle billions of vectors.'), Document(id='cefc95b5-c9ac-4a1c-992b-957916af93da', metadata={'source': 'rag_knowledge_base', 'chunk_id': 10}, page_content='LangChain is a framework for building applications powered by language models. It provides\nabstractions for chains, agents, retrievers, and memory. LangChain integrates with many LLM\nproviders (OpenAI, Anthropic, Groq) and vector stores (FAISS, Chroma, Pinecone). It simplifies\nbuilding RAG pipelines by providin

---

## 2. Why RAG Evaluation Matters

Building a RAG system is one thing - knowing whether it actually works is another. Without evaluation, you cannot:
- Know if your chunking strategy is good
- Know if your retriever is finding the right documents
- Know if the LLM is hallucinating
- Compare two different RAG configurations objectively

**Key evaluation dimensions:**

| Dimension | Question It Answers |
|---|---|
| Faithfulness | Is the answer supported by the retrieved context? (No hallucination?) |
| Answer Relevancy | Does the answer actually address the question? |
| Context Precision | Are the retrieved chunks relevant to the question? |
| Context Recall | Did we retrieve all the information needed to answer? |

---

## 3. RAGAS - RAG Evaluation Framework

RAGAS is the most widely used open-source framework for evaluating RAG pipelines. It computes metrics using an LLM as a judge - this means it works without manually labeled ground truth answers (for most metrics).

**RAGAS Metrics:**

- **Faithfulness** (0-1): Does every claim in the answer have support in the retrieved context? Score of 1 means fully grounded, 0 means hallucinated.
- **Answer Relevancy** (0-1): How well does the answer address the question? Penalizes incomplete or off-topic answers.
- **Context Precision** (0-1): What fraction of retrieved chunks are actually relevant to the question?
- **Context Recall** (0-1): What fraction of the ground truth answer can be attributed to the retrieved context? (Requires reference answer.)

```
RAGAS evaluates the three points of the RAG triangle: Question, Context, and Answer.
Generation Quality (Answer vs Context): Measured by Faithfulness. Does the LLM stick to the facts provided?
Retrieval Quality (Question vs Context): Measured by Context Precision. Did the search engine find the right needles in the haystack?
User Satisfaction (Question vs Answer): Measured by Answer Relevancy. Did the user actually get what they asked for?

The Workflow: How RAGAS Works (The "LLM-as-a-Judge")
RAGAS doesn't just guess; it follows a systematic process for each metric:
Deconstruction: For Faithfulness, RAGAS asks the Judge LLM to break the final answer into individual "claims" (e.g., "The sun is a star," "It is 93 million miles away").
Verification: The Judge LLM then checks every single claim against the retrieved context to see if it's supported.
Calculation: The final score is the percentage of supported claims.
Example: 4 claims made, 3 found in context = 0.75 Faithfulness.

Why use RAGAS? (The Pain Point)
The Problem: Traditional metrics like BLEU or ROUGE only check if words match exactly. They are terrible for AI because they don't understand meaning.
The RAGAS Solution: It uses Semantic Understanding. It knows that "The capital of France" and "Paris is the main city of France" are the same thing, even if the words differ.
No Golden Dataset: You don't need a human to write 1,000 "perfect" answers to test your bot. RAGAS uses the LLM to generate the evaluation.

Summary Table: Which Metric Fixes Which Problem?
            If your RAG is...	                    Look at this Metric:	                        The likely fix:
        Hallucinating (making things up)	          Faithfulness	                    Lower the LLM temperature or improve the prompt.
        Giving "I don't know" answers	              Context Recall     	                Improve your chunking or search algorithm.
        Giving long, rambling answers	              Answer Relevancy	                Update the system prompt to be more concise.
        Finding useless documents	                  Context Precision	                Adjust your embedding model or Top-K value.


```

In [87]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")  

# Initialize the LangChain-compatible Gemini model
# Use "gemini-2.5-flash"
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)


In [88]:
# RAGAS Setup
# RAGAS uses an LLM as a judge to score each metric.
# We use Groq as the judge LLM.

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# Wrap LangChain LLM and embeddings for RAGAS compatibility
# RAGAS needs these wrappers to use our existing LLM and embedding model
# ragas_llm = LangchainLLMWrapper(llm)
ragas_llm = LangchainLLMWrapper(gemini_llm)

ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)

# print("RAGAS configured to use Groq LLM as judge.")
# print("Note: RAGAS calls the LLM multiple times per sample - this will use our Groq quota.")

print("RAGAS configured to use Gemini LLM as judge.")
print("Note: RAGAS calls the LLM multiple times per sample - this will use our Gemini quota.")

RAGAS configured to use Gemini LLM as judge.
Note: RAGAS calls the LLM multiple times per sample - this will use our Gemini quota.


C:\Users\siddhant.khobragade\AppData\Local\Temp\ipykernel_22160\3480674504.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\siddhant.khobragade\AppData\Local\Temp\ipykernel_22160\3480674504.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\siddhant.khobragade\AppData\Local\Temp\ipykernel_22160\3480674504.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\U

In [89]:
# Build the evaluation dataset.
# RAGAS expects a dataset with these fields:
#   - question: the user's query
#   - answer: the generated answer
#   - contexts: list of retrieved text chunks
#   - ground_truth: (optional) reference answer — needed for context_recall only

# Define evaluation questions with ground truth answers
# Ground truth = what the correct answer should be (written by a human expert)
eval_questions = [
    {
        "question": "What is RAG and when was it introduced?",
        "ground_truth": "RAG (Retrieval-Augmented Generation) is an AI framework that retrieves relevant information from an external knowledge base before generating a response. It was introduced by Lewis et al. in 2020."
    },
    {
        "question": "How does RAG reduce hallucination?",
        "ground_truth": "RAG reduces hallucination by grounding the model's response in factual retrieved context instead of relying solely on parametric knowledge."
    },
    {
        "question": "What is FAISS and what company developed it?",
        "ground_truth": "FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta Research for efficient similarity search over large collections of dense vectors."
    },
    {
        "question": "What is the difference between dense and sparse retrieval?",
        "ground_truth": "Dense retrieval uses semantic vector embeddings to find semantically similar documents. Sparse retrieval like BM25 uses keyword frequency matching. Dense captures meaning; sparse handles exact keyword matches."
    },
    {
        "question": "What is reranking and why is it used in RAG?",
        "ground_truth": "Reranking is a post-retrieval technique using a cross-encoder model to rescore retrieved documents. It is used because cross-encoders process query and document together, giving more accurate relevance scores than initial retrieval."
    },
]

# Run RAG pipeline for each question to get answers and contexts
print("Running RAG pipeline on evaluation questions...")
eval_samples = []

for item in eval_questions:
    result = rag_pipeline(item["question"])
    eval_samples.append({
        "question": result["question"],
        "answer": result["answer"],
        "contexts": result["contexts"],
        "ground_truth": item["ground_truth"]
    })
    print(f"  Processed: {item['question'][:60]}...")

print(f"\nEvaluation dataset ready: {len(eval_samples)} samples")

Running RAG pipeline on evaluation questions...
  Processed: What is RAG and when was it introduced?...
  Processed: How does RAG reduce hallucination?...
  Processed: What is FAISS and what company developed it?...
  Processed: What is the difference between dense and sparse retrieval?...
  Processed: What is reranking and why is it used in RAG?...

Evaluation dataset ready: 5 samples


In [91]:
# Convert to HuggingFace Dataset format - required by RAGAS evaluate()

eval_dataset = Dataset.from_list(eval_samples)

print("Dataset schema:")
print(eval_dataset)

# Preview one sample
print("\nSample 1:")
print(f"  Question: {eval_dataset[0]['question']}")
print(f"  Answer: {eval_dataset[0]['answer'][:200]}...")
print(f"  Contexts: {len(eval_dataset[0]['contexts'])} chunks")

Dataset schema:
Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})

Sample 1:
  Question: What is RAG and when was it introduced?
  Answer: I don't know based on the provided information....
  Contexts: 3 chunks


In [92]:
# Run RAGAS evaluation.
# RAGAS will call the judge LLM multiple times for each sample to score each metric.
# This takes 1-3 minutes depending on the number of samples.

# from ragas import RunConfig

# # Configure RAGAS to wait longer and make fewer parallel requests
# # For Groq Free Tier, max_workers=1 or 2 is safest
# run_config = RunConfig(
#     max_workers=2,    # Reduced from 16 to avoid 'Deadline Exceeded'
#     timeout=120,      # Give the model more time to process
#     # max_retries=5,    # Retry if a request fails
#     # max_wait=60       # Wait time between retries
# )

print("Running RAGAS evaluation (this may take a few minutes)...")
print("RAGAS is calling the LLM to judge each metric for each sample.\n")

ragas_results = evaluate(
    dataset=eval_dataset,
    metrics=[
        faithfulness,       # Is the answer grounded in context? (No hallucination)
        answer_relevancy,   # Does the answer address the question?
        context_precision,  # Are retrieved chunks relevant?
        context_recall,     # Was all needed info retrieved? (needs ground_truth)
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    # run_config=run_config, # Pass the config here
    # raise_exceptions=False # Returns NaN instead of crashing if a row fails
)

print("\nRAGAS Evaluation Results:")
print(ragas_results)

Running RAGAS evaluation (this may take a few minutes)...
RAGAS is calling the LLM to judge each metric for each sample.



Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 20/20 [02:23<00:00,  7.18s/it]



RAGAS Evaluation Results:
{'faithfulness': 0.8429, 'answer_relevancy': 0.7254, 'context_precision': 0.6667, 'context_recall': 0.9000}


In [93]:
# Display RAGAS results as a DataFrame for better readability
# Each row is one evaluation sample, each column is one metric

results_df = ragas_results.to_pandas()

print("Per-sample RAGAS scores:")
# print(results_df[["question", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]].to_string(index=False))
print(results_df[["faithfulness", "answer_relevancy", "context_precision", "context_recall"]].to_string(index=False))

print("\nAggregate scores (mean across all samples):")
numeric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
for col in numeric_cols:
    if col in results_df.columns:
        mean_val = results_df[col].mean()
        print(f"  {col}: {mean_val:.4f}")

Per-sample RAGAS scores:
 faithfulness  answer_relevancy  context_precision  context_recall
     0.500000          0.000000           0.000000             0.5
     1.000000          0.785470           0.833333             1.0
     1.000000          0.953009           1.000000             1.0
     1.000000          0.896749           1.000000             1.0
     0.714286          0.991765           0.500000             1.0

Aggregate scores (mean across all samples):
  faithfulness: 0.8429
  answer_relevancy: 0.7254
  context_precision: 0.6667
  context_recall: 0.9000


In [94]:
# Understanding individual RAGAS metrics manually (without the evaluate() wrapper)
# This shows you what RAGAS is actually doing under the hood.

# Faithfulness: RAGAS breaks the answer into atomic claims, then checks if each claim is supported by the retrieved context using the LLM as judge.
#
# Example:
#   Answer: "FAISS was developed by Meta Research in 2017 and supports GPU acceleration."
#   Claims extracted:
#     1. "FAISS was developed by Meta Research" -> Found in context: SUPPORTED
#     2. "FAISS was developed in 2017" -> Not in context: NOT SUPPORTED
#     3. "FAISS supports GPU acceleration" -> Not in context: NOT SUPPORTED
#   Faithfulness = 1/3 = 0.33  (only 1 out of 3 claims is supported)

print("Faithfulness breakdown (conceptual):")
print("  - Answer is broken into atomic claims")
print("  - Each claim is checked against retrieved context")
print("  - Score = supported claims / total claims")

print("\nAnswer Relevancy breakdown (conceptual):")
print("  - RAGAS generates multiple questions from the answer")
print("  - Measures if those questions are similar to the original question")
print("  - High score = answer is on-topic and complete")

print("\nContext Precision breakdown (conceptual):")
print("  - Checks each retrieved chunk: is it relevant to the question?")
print("  - Score = relevant chunks / total retrieved chunks")
print("  - High score = retriever is not pulling irrelevant documents")

print("\nContext Recall breakdown (conceptual):")
print("  - Checks if the ground truth answer can be derived from context")
print("  - Requires ground_truth reference")
print("  - High score = retriever found all needed information")

Faithfulness breakdown (conceptual):
  - Answer is broken into atomic claims
  - Each claim is checked against retrieved context
  - Score = supported claims / total claims

Answer Relevancy breakdown (conceptual):
  - RAGAS generates multiple questions from the answer
  - Measures if those questions are similar to the original question
  - High score = answer is on-topic and complete

Context Precision breakdown (conceptual):
  - Checks each retrieved chunk: is it relevant to the question?
  - Score = relevant chunks / total retrieved chunks
  - High score = retriever is not pulling irrelevant documents

Context Recall breakdown (conceptual):
  - Checks if the ground truth answer can be derived from context
  - Requires ground_truth reference
  - High score = retriever found all needed information


---

## 4. DeepEval - Production LLM Testing Framework

DeepEval provides a pytest-like testing interface for LLM applications. Unlike RAGAS which is focused purely on metrics, DeepEval is designed for ongoing regression testing - you can catch quality degradations when you change your RAG system.

**Key DeepEval concepts:**
- **LLMTestCase**: A single test sample (query + actual output + expected output + context)
- **Metric**: A measurement like HallucinationMetric, FaithfulnessMetric, AnswerRelevancyMetric
- **assert_test()**: Runs the test and fails if any metric is below threshold
- **evaluate()**: Runs all tests and produces a full report

**Why use both RAGAS and DeepEval?**
- RAGAS: Quick batch evaluation, great for exploring metrics
- DeepEval: Structured testing with thresholds, good for CI/CD and regression testing

```
1. Why DeepEval? (The "Safety Net" Philosophy)
If you already have RAGAS, why use DeepEval?
Unit Testing for AI: In traditional coding, you write tests to ensure 2 + 2 = 4. In AI, you can't do that. DeepEval lets you write "Unit Tests" for meaning.
The "Threshold" System: RAGAS gives you a score (e.g., 0.6). DeepEval lets you say: "If the Faithfulness is below 0.7, stop the deployment! This version is too dangerous to release."
Reasoning included: DeepEval is famous for the reason attribute. It doesn't just give a score; it tells you exactly why it failed in plain English (e.g., "The answer mentions '1995' but the context says '1998'.").

2. What is it? (The "G-Eval" Core)
DeepEval primarily uses a technique called G-Eval.
How it works: It feeds your result to a high-end "Judge LLM" (like Llama 3.1 70B or GPT-4) along with a set of scoring criteria.
Multi-Step Judging: It asks the judge to:
Generate steps to evaluate the metric.
Assign weights to those steps.
Provide a final score based on the weightage.
This makes it more human-like than simple mathematical formulas.

3. How does it work? (The Execution Cycle)
When you run assert_test(test_case, [metrics]), this is what happens behind the scenes:
Ingestion: It takes your input, actual_output, and retrieval_context.
Comparison: It checks for Contradictions (Hallucination) and Alignment (Relevancy).
Threshold Check: It compares the score against your threshold=0.7.
Logging: If the test fails, it logs the Reason so you can fix your prompt or data immediately.

RAGAS vs. DeepEval: The Professional Workflow
In a real-world company, you use them in this order:
Development (RAGAS): You use RAGAS to experiment with different chunk sizes and embedding models to see which scores higher on average.
Production/CI-CD (DeepEval): Once you are happy, you create 50 "Golden Test Cases" in DeepEval. Every time you update your code, you run these tests. If even one test fails the threshold, you don't push the code to users.

```

In [97]:
# DeepEval Setup
# DeepEval uses an LLM as judge too. We configure it to use Groq.

from deepeval import evaluate as deepeval_evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    HallucinationMetric,
    AnswerRelevancyMetric,
    FaithfulnessMetric,
)
from deepeval.models.base_model import DeepEvalBaseLLM
from langchain_groq import ChatGroq

# DeepEval requires a custom LLM wrapper class to use non-OpenAI models
# This wrapper connects DeepEval's judge to our Groq LLM

class GroqDeepEvalModel(DeepEvalBaseLLM):
    """
    Custom DeepEval LLM wrapper that connects to Groq.
    DeepEval uses this as the judge for all metrics.
    """
    
    def __init__(self):
        self.groq_llm = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0.0,  # Deterministic for consistent evaluation
            max_tokens=1000,
        )
    
    def load_model(self):
        return self.groq_llm
    
    def generate(self, prompt: str) -> str:
        messages = [HumanMessage(content=prompt)]
        response = self.groq_llm.invoke(messages)
        return response.content
    
    async def a_generate(self, prompt: str) -> str:
        # Async version - falls back to sync for simplicity
        return self.generate(prompt)
    
    def get_model_name(self) -> str:
        return "groq-llama-3.1-8b-instant"


groq_judge = GroqDeepEvalModel()
print("DeepEval configured with Groq as judge LLM.")

DeepEval configured with Groq as judge LLM.


In [98]:
# Define DeepEval metrics with thresholds.
# Threshold: if a metric score drops below this value, the test fails.
# This is useful for CI/CD - if you change your RAG system, run these tests to catch regressions.

# HallucinationMetric: checks if the actual output contains information
# NOT present in the provided context (hallucination)
hallucination_metric = HallucinationMetric(
    threshold=0.5,      # Fail if hallucination score > 0.5 (less than 50% hallucinated)
    model=groq_judge,
    include_reason=True  # Include explanation of why score was given
)

# AnswerRelevancyMetric: checks if the actual output addresses the input question
answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,      # Fail if relevancy score < 0.7
    model=groq_judge,
    include_reason=True
)

# FaithfulnessMetric: checks if the answer is factually consistent with retrieved context
faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,      # Fail if faithfulness < 0.7
    model=groq_judge,
    include_reason=True
)

print("DeepEval metrics defined:")
print(f"  HallucinationMetric (threshold=0.5 -> lower is better)")
print(f"  AnswerRelevancyMetric (threshold=0.7 -> higher is better)")
print(f"  FaithfulnessMetric (threshold=0.7 -> higher is better)")

DeepEval metrics defined:
  HallucinationMetric (threshold=0.5 -> lower is better)
  AnswerRelevancyMetric (threshold=0.7 -> higher is better)
  FaithfulnessMetric (threshold=0.7 -> higher is better)


In [104]:
# Create DeepEval test cases from our RAG results.
# LLMTestCase is the basic unit in DeepEval - it captures one interaction.

# Run our RAG pipeline on evaluation questions
deepeval_test_cases = []

test_questions = [
    "What is RAG and when was it introduced?",
    "How does RAG reduce hallucination?",
    "What is FAISS?",
]

for question in test_questions:
    result = rag_pipeline(question)
    
    # LLMTestCase requires:
    # - input: the user's question
    # - actual_output: what the RAG system generated
    # - retrieval_context: the chunks retrieved (for faithfulness + hallucination metrics)
    test_case = LLMTestCase(
        input=result["question"],
        actual_output=result["answer"],
        retrieval_context=result["contexts"],  # Retrieved chunks
        context=result["contexts"],           # for HallucinationMetric
    )
    deepeval_test_cases.append(test_case)

print(f"Created {len(deepeval_test_cases)} DeepEval test cases.")

# Preview a test case
tc = deepeval_test_cases[0]
print(f"\nTest Case 1:")
print(f"  Input: {tc.input}")
print(f"  Actual Output: {tc.actual_output[:200]}...")
print(f"  Retrieved context chunks: {len(tc.retrieval_context)}")

Created 3 DeepEval test cases.

Test Case 1:
  Input: What is RAG and when was it introduced?
  Actual Output: I don't know based on the provided information....
  Retrieved context chunks: 3


In [105]:
# Run DeepEval evaluation on a single test case
# We test one at a time here so we can see the reasoning for each metric.

from deepeval import assert_test

# Evaluate the first test case with all three metrics
test_case = deepeval_test_cases[0]

print(f"Evaluating test case: '{test_case.input}'")
print("Running three metrics (each calls the LLM judge)...\n")

# Score each metric individually
hallucination_metric.measure(test_case)
answer_relevancy_metric.measure(test_case)
faithfulness_metric.measure(test_case)

print("Results:")
print(f"  Hallucination Score: {hallucination_metric.score:.4f}")
print(f"  Hallucination Reason: {hallucination_metric.reason}")
print()
print(f"  Answer Relevancy Score: {answer_relevancy_metric.score:.4f}")
print(f"  Answer Relevancy Reason: {answer_relevancy_metric.reason}")
print()
print(f"  Faithfulness Score: {faithfulness_metric.score:.4f}")
print(f"  Faithfulness Reason: {faithfulness_metric.reason}")

Evaluating test case: 'What is RAG and when was it introduced?'
Running three metrics (each calls the LLM judge)...



d:\Work\Learning\Topics_by_Abhay_Sir\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Results:
  Hallucination Score: 1.0000
  Hallucination Reason: The score is 1.00 because the actual output contradicts the provided context in multiple places, indicating a high level of hallucination.

  Answer Relevancy Score: 0.5000
  Answer Relevancy Reason: The score is 0.50 because the actual output included irrelevant statements, such as 'I don't know indicates a lack of knowledge, not a direct answer to the question.', which detract from the relevance of the answer to the question 'What is RAG and when was it introduced?'

  Faithfulness Score: 0.8333
  Faithfulness Reason: The score is 0.83 because the AI's output was mostly faithful to the retrieval context, but there was a minor contradiction where the AI expressed uncertainty based on the given text, indicating a slight deviation from the expected output.


In [106]:
# Run DeepEval on all test cases together using the evaluate() function.
# This produces a structured report showing pass/fail for each test.

print("Running full DeepEval batch evaluation...")
print("Each test case is scored on all three metrics.\n")

# evaluate() runs all test cases against all specified metrics
# It returns a list of results and prints a summary table
deepeval_results = deepeval_evaluate(
    test_cases=deepeval_test_cases,
    metrics=[
        hallucination_metric,
        answer_relevancy_metric,
        faithfulness_metric,
    ]
)

print("\nDeepEval evaluation complete.")

Running full DeepEval batch evaluation...
Each test case is scored on all three metrics.



✨ You're running DeepEval's latest Hallucination Metric! (using groq-llama-3.1-8b-instant, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using groq-llama-3.1-8b-instant, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using groq-llama-3.1-8b-instant, strict=False, 
async_mode=True)...

Warning: Could not load test run from disk: pywintypes is required for Win32Locker but not found. Please install 
pywin32.

...

Warning: Could not load test run from disk: pywintypes is required for Win32Locker but not found. Please install 
pywin32.

In wrap_up_cached_test_run, Error saving test run to disk, pywintypes is required for Win32Locker but not found. Please install pywin32.


⚠ WARNING: No hyperparameters logged.
» ]8;id=727643;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 106.59s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 66.67% | Passed: 2 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


DeepEval evaluation complete.


In [107]:
# Using assert_test() for pass/fail testing — like pytest for LLMs
# This is the pattern to use in CI/CD pipelines.
# If your RAG changes cause a score to drop below threshold, the test fails.

def run_rag_quality_tests(questions):
    """
    Run quality tests on a set of questions.
    Returns True if all tests pass, False if any fail.
    Think of this as your test suite that runs whenever you update the RAG system.
    """
    all_passed = True
    
    for question in questions:
        result = rag_pipeline(question)
        
        test_case = LLMTestCase(
            input=result["question"],
            actual_output=result["answer"],
            retrieval_context=result["contexts"],
        )
        
        # Score the metrics
        answer_relevancy_metric.measure(test_case)
        faithfulness_metric.measure(test_case)
        
        # Check if they pass the threshold
        relevancy_passed = answer_relevancy_metric.score >= 0.7
        faith_passed = faithfulness_metric.score >= 0.7
        
        status = "PASS" if (relevancy_passed and faith_passed) else "FAIL"
        if status == "FAIL":
            all_passed = False
        
        print(f"[{status}] '{question[:60]}...'")
        print(f"       Relevancy: {answer_relevancy_metric.score:.3f} (threshold=0.7)")
        print(f"       Faithfulness: {faithfulness_metric.score:.3f} (threshold=0.7)")
    
    return all_passed


# Run the quality test suite
test_q = [
    "What are the four components of a RAG pipeline?",
    "What embedding dimensions does all-MiniLM-L6-v2 produce?",
]

print("Running RAG quality test suite...\n")
all_pass = run_rag_quality_tests(test_q)
print(f"\nOverall result: {'ALL TESTS PASSED' if all_pass else 'SOME TESTS FAILED'}")

Running RAG quality test suite...



[PASS] 'What are the four components of a RAG pipeline?...'
       Relevancy: 1.000 (threshold=0.7)
       Faithfulness: 1.000 (threshold=0.7)


[PASS] 'What embedding dimensions does all-MiniLM-L6-v2 produce?...'
       Relevancy: 1.000 (threshold=0.7)
       Faithfulness: 1.000 (threshold=0.7)

Overall result: ALL TESTS PASSED


---

## 5. Building a Production RAG Chatbot

Now we bring everything together into a complete, production-ready RAG chatbot. This chatbot will have:

- Multi-document ingestion (PDF, text, etc.)
- Persistent vector store (save to disk, load without re-embedding)
- Conversation memory (multi-turn dialogue)
- Source citation (shows which chunks were used)
- Streaming output (real-time text generation)
- Interactive terminal chat loop

In [109]:
# Component 1: Multi-Document Ingestion
# Supports .txt and .pdf files (extend as needed)
# Processes each file, chunks it, and adds to a unified knowledge base.

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader
import os

def ingest_documents(file_paths, chunk_size=400, chunk_overlap=80):
    """
    Load and chunk multiple documents from disk.
    Supports .txt and .pdf files.
    Returns a list of LangChain Document objects ready for embedding.
    """
    all_chunks = []
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    
    for file_path in file_paths:
        ext = os.path.splitext(file_path)[1].lower()
        
        # Load the document based on file type
        if ext == ".txt":
            loader = TextLoader(file_path, encoding="utf-8")
        elif ext == ".pdf":
            loader = PyPDFLoader(file_path)
        else:
            print(f"Skipping unsupported file type: {file_path}")
            continue
        
        docs = loader.load()
        
        # Split into chunks and add source metadata
        chunks = splitter.split_documents(docs)
        for i, chunk in enumerate(chunks):
            chunk.metadata["source"] = os.path.basename(file_path)
            chunk.metadata["chunk_id"] = f"{os.path.basename(file_path)}_{i}"
        
        all_chunks.extend(chunks)
        print(f"Ingested '{file_path}': {len(chunks)} chunks")
    
    print(f"\nTotal chunks ingested: {len(all_chunks)}")
    return all_chunks


# For this demo, we create a sample .txt file and ingest it
sample_doc_path = "rag_knowledge.txt"

with open(sample_doc_path, "w") as f:
    f.write(knowledge_base_text)

ingested_chunks = ingest_documents([sample_doc_path])

Ingested 'rag_knowledge.txt': 11 chunks

Total chunks ingested: 11


In [110]:
# Component 2: Persistent Vector Store
# FAISS can save its index to disk and reload it without re-embedding.
# This is critical in production: you only embed documents once, then load the saved index on each app startup.

from langchain_community.vectorstores import FAISS
import os

VECTOR_STORE_PATH = "./faiss_index"  # Directory where FAISS index will be saved

def build_and_save_vectorstore(chunks, embedding_model, save_path):
    """
    Build FAISS from document chunks and save to disk.
    Run this once when you first set up your knowledge base.
    """
    print("Building FAISS vector store...")
    vs = FAISS.from_documents(chunks, embedding_model)
    
    # Save index to disk — creates two files: index.faiss and index.pkl
    vs.save_local(save_path)
    print(f"Vector store saved to '{save_path}'")
    return vs


def load_vectorstore(save_path, embedding_model):
    """
    Load a previously saved FAISS vector store from disk.
    Much faster than rebuilding from scratch on each startup.
    """
    vs = FAISS.load_local(
        save_path,
        embedding_model,
        allow_dangerous_deserialization=True  # Required for loading FAISS pickled files
    )
    print(f"Vector store loaded from '{save_path}'")
    return vs


# Build and save the vector store
if not os.path.exists(VECTOR_STORE_PATH):
    chatbot_vs = build_and_save_vectorstore(ingested_chunks, embedding_model, VECTOR_STORE_PATH)
else:
    print(f"Existing vector store found at '{VECTOR_STORE_PATH}' — loading from disk.")
    chatbot_vs = load_vectorstore(VECTOR_STORE_PATH, embedding_model)

print("Vector store ready.")

Building FAISS vector store...
Vector store saved to './faiss_index'
Vector store ready.


In [113]:
# Component 3: Streaming Response Generation
# Instead of waiting for the full response, stream tokens as they are generated.
# This is how real chatbots feel responsive - the user sees output appearing in real time.

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# Streaming LLM - same model but with streaming=True
streaming_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=800,
    streaming=True,  # Enable token-by-token streaming
)


def generate_streaming_response(messages):
    """
    Generates a response and streams it token by token.
    Returns the full accumulated response text.
    """
    print("Assistant: ", end="", flush=True)
    full_response = ""
    
    # stream() yields chunks (AIMessageChunk) one by one
    for chunk in streaming_llm.stream(messages):
        token = chunk.content  # Each chunk is a partial token
        print(token, end="", flush=True)  # Print without newline, flush immediately
        full_response += token
    
    print()  # Newline after streaming is complete
    return full_response


# Test streaming
test_messages = [
    SystemMessage(content="Answer concisely."),
    HumanMessage(content="What is RAG in one sentence?")
]

print("Streaming test:")
_ = generate_streaming_response(test_messages)

Streaming test:
Assistant: RAG stands for Reach, Achieve, Grow, which is a framework often used in project management and personal development to set goals and track progress.


In [114]:
# Component 4: The Full RAG Chatbot Class
# Combines all components: persistent vector store, conversation memory, source citations, and streaming output.

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from typing import List, Tuple

class RAGChatbot:
    """
    A production-ready RAG chatbot with:
    - Semantic retrieval from FAISS vector store
    - Conversation memory across turns
    - Source citations with chunk references
    - Streaming output
    - Session management (reset)
    """
    
    def __init__(
        self,
        vectorstore: FAISS,
        llm,
        k: int = 3,
        max_history: int = 6,
        stream: bool = True
    ):
        self.vectorstore = vectorstore
        self.llm = llm
        self.k = k
        self.max_history = max_history  # Max conversation turns to keep in context
        self.stream = stream
        self.chat_history: List[Tuple[str, str]] = []  # (user, assistant) pairs
        
        # System prompt — sets the chatbot's behaviour
        self.system_prompt = (
            "You are a knowledgeable assistant with access to a curated knowledge base. "
            "Answer questions using the provided context. If the answer isn't in the context, "
            "say so clearly - do not fabricate information. "
            "Be concise, precise, and cite relevant facts from the context."
        )
    
    def retrieve(self, query: str):
        """Retrieve top-k relevant chunks from the vector store."""
        docs = self.vectorstore.similarity_search(query, k=self.k)
        return docs
    
    def build_messages(self, query: str, context: str) -> List:
        """
        Construct the full message list:
        [system] + [recent chat history] + [current query with context]
        """
        messages = [SystemMessage(content=self.system_prompt)]
        
        # Add recent conversation history
        recent = self.chat_history[-self.max_history:]
        for human_msg, ai_msg in recent:
            messages.append(HumanMessage(content=human_msg))
            messages.append(AIMessage(content=ai_msg))
        
        # Add current query with retrieved context
        messages.append(HumanMessage(
            content=f"Knowledge Base Context:\n{context}\n\nQuestion: {query}"
        ))
        
        return messages
    
    def chat(self, user_query: str, verbose: bool = True) -> dict:
        """
        Main chat method. Processes a user query and returns the response.
        
        Args:
            user_query: The user's question
            verbose: If True, print streaming output and sources
        
        Returns:
            dict with 'answer', 'sources', 'context_chunks'
        """
        # Step 1: Retrieve relevant chunks
        docs = self.retrieve(user_query)
        context = "\n\n".join([doc.page_content for doc in docs])
        
        # Step 2: Build message list
        messages = self.build_messages(user_query, context)
        
        # Step 3: Generate response (streaming or non-streaming)
        if self.stream and verbose:
            full_answer = generate_streaming_response(messages)
        else:
            response = self.llm.invoke(messages)
            full_answer = response.content
            if verbose:
                print(f"Assistant: {full_answer}")
        
        # Step 4: Extract source references
        sources = []
        for doc in docs:
            source_info = {
                "source": doc.metadata.get("source", "Unknown"),
                "chunk_id": doc.metadata.get("chunk_id", "N/A"),
                "preview": doc.page_content[:100] + "..."
            }
            sources.append(source_info)
        
        # Step 5: Save to conversation history
        self.chat_history.append((user_query, full_answer))
        
        return {
            "answer": full_answer,
            "sources": sources,
            "context_chunks": docs
        }
    
    def print_sources(self, result: dict):
        """Print the source citations for the last response."""
        print("\nSources used:")
        for i, s in enumerate(result["sources"]):
            print(f"  [{i+1}] {s['source']} | Chunk: {s['chunk_id']}")
            print(f"      Preview: {s['preview']}")
    
    def reset(self):
        """Start a new conversation — clears all history."""
        self.chat_history = []
        print("New conversation started.")
    
    def summary(self):
        """Print a summary of the current conversation."""
        print(f"Conversation turns: {len(self.chat_history)}")
        print(f"Retrieval k: {self.k}")
        print(f"Max history turns kept: {self.max_history}")


# Initialize the chatbot
chatbot = RAGChatbot(
    vectorstore=chatbot_vs,
    llm=streaming_llm,
    k=3,
    max_history=6,
    stream=True
)

print("RAG Chatbot initialized and ready.")

RAG Chatbot initialized and ready.


In [115]:
# Test the RAG Chatbot with a multi-turn conversation

print("=" * 60)
print("RAG Chatbot Demo — Multi-Turn Conversation")
print("=" * 60)

# Turn 1: Factual question
print("\n[Turn 1]")
print("User: What is FAISS and how does it relate to RAG?")
result1 = chatbot.chat("What is FAISS and how does it relate to RAG?")
chatbot.print_sources(result1)

# Turn 2: Follow-up referencing previous answer
print("\n[Turn 2]")
print("User: Can it handle billions of vectors?")
result2 = chatbot.chat("Can it handle billions of vectors?")  # 'it' refers to FAISS from Turn 1
chatbot.print_sources(result2)

# Turn 3: New topic
print("\n[Turn 3]")
print("User: What is reranking and why is it important?")
result3 = chatbot.chat("What is reranking and why is it important?")
chatbot.print_sources(result3)

print("\n" + "=" * 60)
chatbot.summary()

RAG Chatbot Demo — Multi-Turn Conversation

[Turn 1]
User: What is FAISS and how does it relate to RAG?
Assistant: FAISS (Facebook AI Similarity Search) is not mentioned in the provided context. However, based on general knowledge, FAISS is an open-source library for efficient similarity search and clustering of dense vectors. 

In the context of RAG, the retriever component can use embedding similarity for finding relevant documents. FAISS is often used for efficient similarity search, which can be applied in the retriever component of RAG for efficient document retrieval based on embedding similarity. However, this is not explicitly mentioned in the provided context.

Sources used:
  [1] rag_knowledge.txt | Chunk: rag_knowledge.txt_9
      Preview: RAGAS (Retrieval-Augmented Generation Assessment) is an evaluation framework for RAG systems.
It pro...
  [2] rag_knowledge.txt | Chunk: rag_knowledge.txt_2
      Preview: RAG reduces hallucination by grounding the model's response in fact

In [117]:
# Component 5: Interactive Terminal Chat Loop
# This is a runnable chatbot interface in the terminal.
# Type 'exit' to quit, 'reset' to start a new conversation, 'sources' to see citations.

def run_chatbot_cli():
    """
    Interactive command-line chatbot.
    Commands:
      exit   -> Quit the chatbot
      reset  -> Start a new conversation
      sources -> Show sources from the last answer
    """
    chatbot.reset()
    last_result = None
    
    print("\n" + "=" * 60)
    print("RAG Chatbot - Interactive Mode")
    print("Commands: 'exit' | 'reset' | 'sources'")
    print("=" * 60 + "\n")
    
    while True:
        # Get user input
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting chatbot.")
            break
        
        if not user_input:
            continue
        
        # Handle commands
        if user_input.lower() == "exit":
            print("Goodbye!")
            break
        
        elif user_input.lower() == "reset":
            chatbot.reset()
            last_result = None
            continue
        
        elif user_input.lower() == "sources":
            if last_result:
                chatbot.print_sources(last_result)
            else:
                print("No previous answer to show sources for.")
            continue
        
        # Normal question — run RAG pipeline
        last_result = chatbot.chat(user_input)
        print()  # Blank line for readability between turns


# Run the interactive chatbot
# Uncomment the line below to start the interactive chat loop
run_chatbot_cli()

print("Interactive chatbot function defined.")
print("To start the interactive chat, uncomment and run: run_chatbot_cli()")

New conversation started.

RAG Chatbot - Interactive Mode
Commands: 'exit' | 'reset' | 'sources'



Assistant: I'm ready to assist you. However, I don't have any specific information to draw from in our conversation yet. To get started, could you please ask a question or provide more context about what you'd like to discuss?

Assistant: RAG stands for Retrieval-Augmented Generation. It's an evaluation framework for RAG systems, but more broadly, it's a pipeline that reduces hallucination in Large Language Models (LLMs) by grounding their responses in factual, retrieved context.

Assistant: You asked me about "RAG" and I provided an initial answer. However, after you provided the knowledge base context, I realized that RAG stands for Retrieval-Augmented Generation, an AI framework that retrieves relevant information from an external knowledge base before generating a response.

Assistant: Based on the provided context, RAG (Retrieval-Augmented Generation) reduces hallucination by grounding the model's response in factual, retrieved context, which includes retrieved documents. This all

In [118]:
# Bonus: Update the knowledge base by adding new documents to an existing vector store.
# This is how you extend the chatbot's knowledge without rebuilding from scratch.

def add_to_knowledge_base(new_file_paths, vectorstore, embedding_model, save_path):
    """
    Add new documents to an existing vector store.
    
    This is the incremental update pattern:
    - Load new documents
    - Chunk them
    - Add embeddings to existing FAISS index
    - Save the updated index
    
    You do NOT need to re-embed all existing documents.
    """
    # Ingest new documents
    new_chunks = ingest_documents(new_file_paths)
    
    if not new_chunks:
        print("No new chunks to add.")
        return vectorstore
    
    # Add new chunks to the existing vector store
    # FAISS.add_documents() embeds new docs and adds them to the existing index
    vectorstore.add_documents(new_chunks)
    
    # Save the updated index
    vectorstore.save_local(save_path)
    
    print(f"Added {len(new_chunks)} new chunks. Index saved.")
    return vectorstore


# Example usage:
#
new_doc_path = "data/sample.txt"
chatbot_vs = add_to_knowledge_base(
    new_file_paths=[new_doc_path],
    vectorstore=chatbot_vs,
    embedding_model=embedding_model,
    save_path=VECTOR_STORE_PATH
)
# Update chatbot to use the new vector store
chatbot.vectorstore = chatbot_vs

print("Incremental knowledge base update function defined.")

Ingested 'data/sample.txt': 42 chunks

Total chunks ingested: 42
Added 42 new chunks. Index saved.
Incremental knowledge base update function defined.


---

## 6. Summary

This notebook completed the RAG learning journey with evaluation and production deployment.

**RAGAS Evaluation:**
- Faithfulness: Are claims in the answer supported by context? (Detects hallucination)
- Answer Relevancy: Does the answer address the question?
- Context Precision: Are retrieved chunks relevant?
- Context Recall: Was all needed information retrieved?

**DeepEval Testing:**
- LLMTestCase: Wraps a single interaction for testing
- HallucinationMetric, AnswerRelevancyMetric, FaithfulnessMetric: Scored against thresholds
- assert_test() / evaluate(): Pass/fail testing for CI/CD pipelines

**Production RAG Chatbot:**
- Multi-document ingestion: .txt, .pdf support with source metadata
- Persistent FAISS: Save once, load every time - no repeated embedding
- Streaming output: Real-time token generation
- Conversation memory: Multi-turn dialogue with context awareness
- Source citation: Every answer shows which chunks were used
- Incremental updates: Add new documents without rebuilding

---

**Full RAG Learning Path Completed:**

Notebook 1 -> Basic RAG: Setup, chunking, FAISS, prompting, citations

Notebook 2 -> Advanced RAG: Semantic/hierarchical chunking, HyDE, multi-query, branched RAG, hybrid search, reranking

Notebook 3 -? Evaluation + Chatbot: RAGAS, DeepEval, streaming, persistent store, full chatbot